# Training Walkthrough: Binary Segmentation with `segmentation_tools_pt`

> **What you will learn:**  
> Build a complete training pipeline — from raw images to a deployed ONNX model — using every module of this library. All data is synthesized inside the notebook, so this runs on the Linux HPC with no external files.

---

## Channel setups supported

| Mode | Source files on disk | Model input (`build_image`) | `in_channels` |
|------|---------------------|------------------------------|---------------|
| **single** | `images/{id}.png` + `masks/{id}.png` | 1 channel: raw grayscale | 1 |
| **single_to_3** | same as single | 3 channels: raw + top-hat + Sobel edges | 3 |
| **two_channel** | `{id}_ch0.png`, `{id}_ch1.png` | 3 channels: ch0 + ch1 + product | 3 |
| **three_channel** | `{id}_ch0.png`, `_ch1`, `_ch2` | 3 channels: pass-through | 3 |

Set `DATA_MODE` in Step 0 — every downstream cell reads from that one config block.

---

## Overview

| Step | What happens | Module used |
|------|-------------|-------------|
| 0 | Setup + **channel config** | `core`, `channels` |
| 1 | Synthesize a raw image dataset | — |
| 2 | Inspect images, masks, channel recipe | `channels`, `masks` |
| 3 | Build the patch dataset on disk | `patches`, `dataset_build` |
| 4 | Compute per-channel statistics | `transforms` |
| 5 | Sanity-check the augmented dataset | `dataset`, `visualize` |
| 6 | Build the model | `model` |
| 7 | Train | `losses`, `metrics`, `train` |
| 8 | Evaluate on validation | `metrics`, `visualize` |
| 9 | Find the operating threshold | `inference` |
| 10 | Export to ONNX | `export` |

---

## Step 0 — Setup

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2


In [ ]:
# Standard library
import json
from pathlib import Path

# Third-party
import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

# ---- segmentation_tools_pt ----
from segmentation_tools_pt.core          import set_seed, get_device, SegConfig, platform_info
from segmentation_tools_pt.channels      import build_image, CHANNEL_OPS, DEFAULT_CHANNELS, tophat
from segmentation_tools_pt.masks         import binarize, fit_rotated_rects_mask
from segmentation_tools_pt.patches       import PatchConfig
from segmentation_tools_pt.dataset_build import build_patch_dataset
from segmentation_tools_pt.transforms    import compute_dataset_stats
from segmentation_tools_pt.dataset       import build_datasets, build_dataloaders
from segmentation_tools_pt.losses        import SegLoss
from segmentation_tools_pt.model         import get_model, get_param_groups
from segmentation_tools_pt.train         import train_one_epoch, validate, save_checkpoint, load_checkpoint
from segmentation_tools_pt.inference     import predict, decode, find_operating_threshold
from segmentation_tools_pt.visualize     import sanity_check, show_pred, show_batch, show_top_losses
from segmentation_tools_pt.export        import export_onnx

print(f'torch {torch.__version__} | cuda {torch.cuda.is_available()}')

In [ ]:
set_seed(42)
DEVICE = get_device()

# Always run this first on a new machine — catches platform-specific issues early
for k, v in platform_info().items():
    print(f'{k:<20}: {v}')


In [ ]:

# =============================================================================
# DATA_MODE — pick ONE; all cells below read these variables
# =============================================================================
# 'single'        : one grayscale file per sample  → 1-channel model input
# 'single_to_3'   : one file, derive 3 channels (raw + top-hat + Sobel)
# 'two_channel'   : two captures (diffuse + specular) → 3 channels via product
# 'three_channel' : three native grayscale channels → 3-channel pass-through
DATA_MODE = 'single'

# Extra channel ops for single→3 and native ch2
CHANNEL_OPS['ch2'] = lambda imgs: imgs[2]
CHANNEL_OPS['tophat_ch0'] = lambda imgs: tophat(imgs[0], 21)
CHANNEL_OPS['sobel_ch0'] = lambda imgs: cv2.convertScaleAbs(
    cv2.Sobel(imgs[0], cv2.CV_32F, 1, 1, ksize=3))

CHANNEL_PRESETS = {
    'single': dict(
        n_source_channels=1,
        channel_names=['ch0'],
        in_channels=1,
        mask_suffix='',           # masks/{id}.png  (no _mask suffix)
    ),
    'single_to_3': dict(
        n_source_channels=1,
        channel_names=['ch0', 'tophat_ch0', 'sobel_ch0'],
        in_channels=3,
        mask_suffix='',
    ),
    'two_channel': dict(
        n_source_channels=2,
        channel_names=['ch0', 'ch1', 'product'],
        in_channels=3,
        mask_suffix='_mask',
    ),
    'three_channel': dict(
        n_source_channels=3,
        channel_names=['ch0', 'ch1', 'ch2'],
        in_channels=3,
        mask_suffix='_mask',
    ),
}


In [ ]:
cfg_ch = CHANNEL_PRESETS[DATA_MODE]
cfg_ch

In [ ]:
N_SOURCE_CHANNELS = cfg_ch['n_source_channels']
N_SOURCE_CHANNELS

In [ ]:
CHANNEL_NAMES     = cfg_ch['channel_names']
CHANNEL_NAMES

In [ ]:
IN_CHANNELS       = cfg_ch['in_channels']
IN_CHANNELS

In [ ]:
MASK_SUFFIX       = cfg_ch['mask_suffix']
MASK_SUFFIX

In [ ]:
CFG = SegConfig(in_channels=IN_CHANNELS, num_classes=1)
CFG

In [ ]:
print(f'DATA_MODE          : {DATA_MODE}')
print(f'n_source_channels  : {N_SOURCE_CHANNELS}')
print(f'channel_names      : {CHANNEL_NAMES}')
print(f'model in_channels  : {IN_CHANNELS}')
print(f'mask_suffix        : {MASK_SUFFIX!r}')
print('SegConfig          :', CFG)

---
## Step 1 — Create a synthetic raw dataset

The on-disk layout depends on `DATA_MODE` (set in Step 0):

| Mode | `images/` | `masks/` |
|------|-----------|----------|
| `single`, `single_to_3` | `{id}.png` | `{id}.png` |
| `two_channel`, `three_channel` | `{id}_ch0.png`, `{id}_ch1.png`, … | `{id}_mask.png` or `{id}.png` |

> **On real data:** copy your files into the matching layout, set `DATA_MODE`, and re-run from Step 1.  
> Example — single-channel X-ray: `DATA_MODE='single'`, `mask_suffix=''`.

In [ ]:
# ---- tuneable constants -----------------------------------------------
N_SAMPLES       = 8       # number of source images
IMAGE_H, IMAGE_W = 256, 256  # source image size (px)
N_RECTS_PER_IMG = 4       # rectangular targets per image
RECT_MIN, RECT_MAX = 20, 45  # target side length range (px)
# -----------------------------------------------------------------------

WORK_DIR = Path('tutorial_run')   # all outputs land here
RAW_DIR  = WORK_DIR / 'raw'
(RAW_DIR / 'images').mkdir(parents=True, exist_ok=True)
(RAW_DIR / 'masks').mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(0)

for i in range(N_SAMPLES):
    sid = f'sample_{i:02d}'

    # Diffuse channel: uniform noisy background
    ch0 = rng.integers(60, 120, (IMAGE_H, IMAGE_W), dtype=np.uint8)
    # Specular channel: same background but targets are brighter (simulate specular highlight)
    ch1 = rng.integers(40, 100, (IMAGE_H, IMAGE_W), dtype=np.uint8)
    mask = np.zeros((IMAGE_H, IMAGE_W), np.uint8)

    for _ in range(N_RECTS_PER_IMG):
        h_rect = int(rng.integers(RECT_MIN, RECT_MAX))
        w_rect = int(rng.integers(RECT_MIN, RECT_MAX))
        y0 = int(rng.integers(5, IMAGE_H - h_rect - 5))
        x0 = int(rng.integers(5, IMAGE_W - w_rect - 5))
        # Targets are brighter in both channels
        ch0[y0:y0+h_rect, x0:x0+w_rect] = rng.integers(160, 220, (h_rect, w_rect), np.uint8)
        ch1[y0:y0+h_rect, x0:x0+w_rect] = rng.integers(180, 240, (h_rect, w_rect), np.uint8)
        mask[y0:y0+h_rect, x0:x0+w_rect] = 255

    cv2.imwrite(str(RAW_DIR / 'images' / f'{sid}_ch0.png'), ch0)
    cv2.imwrite(str(RAW_DIR / 'images' / f'{sid}_ch1.png'), ch1)
    cv2.imwrite(str(RAW_DIR / 'masks'  / f'{sid}_mask.png'), mask)

print(f'Created {N_SAMPLES} synthetic samples in {RAW_DIR}')

In [ ]:
RAW_DIR=Path(r'/path/to/dataset/segmentation')


---
## Step 1b — Connect to real data and validate the folder

> **Skip this step if you are using the synthetic dataset above.**  
> Run this section before anything else when `RAW_DIR` points at your real images.

It does four things:
1. **Counts** how many samples are discoverable (images + matching masks).
2. **Checks mask encoding** — detects `0/1` vs `0/255` and warns if foreground pixels are zero (the silent killer of `fg_pixels`).
3. **Checks size consistency** — all images and masks must be the same shape.
4. **Visual spot-check** — plots a random selection of image + mask pairs so you confirm label quality before spending any GPU time.

In [ ]:
from segmentation_tools_pt.dataset_build import discover_samples, resolve_channel_paths, resolve_mask_path

# ── 1. Discover samples ──────────────────────────────────────────────────────
try:
    sample_ids = discover_samples(str(RAW_DIR), n_channels=N_SOURCE_CHANNELS, mask_suffix=MASK_SUFFIX)
    print(f'Found {len(sample_ids)} samples in {RAW_DIR}')
    print(f'First 5 : {sample_ids[:5]}')
    print(f'Last  5 : {sample_ids[-5:]}')
except FileNotFoundError as e:
    print(f'ERROR: {e}')
    print('Check that RAW_DIR, DATA_MODE and MASK_SUFFIX match your folder layout.')

In [ ]:
# ── 2. Mask encoding check ───────────────────────────────────────────────────
# Masks must be EITHER {0,255}  OR  {0,1}.
# If all values are 0 the mask is blank — patches will have fg_pixels=0 and training fails.

problems = []
for sid in sample_ids[:20]:          # check the first 20; adjust to check all
    mask_path = resolve_mask_path(Path(RAW_DIR), sid, mask_suffix=MASK_SUFFIX)
    m = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    uniq = np.unique(m).tolist()
    if m.max() == 0:
        problems.append((sid, 'BLANK mask — all zeros', uniq))
    elif not set(uniq).issubset({0, 1, 255}):
        problems.append((sid, f'unexpected values (multi-class? use DATA_MODE=two_channel + num_classes=K)', uniq))

if problems:
    print(f'\n{"Sample ID":<50}  {"Issue":<60}  {"Unique values"}')
    print('-' * 130)
    for sid, msg, uniq in problems:
        print(f'{sid:<50}  {msg:<60}  {uniq}')
else:
    print(f'OK — all {min(20, len(sample_ids))} checked masks have valid encoding.')

# Show the encoding distribution across all checked samples
from collections import Counter
enc_counts = Counter()
for sid in sample_ids[:50]:
    m = cv2.imread(str(resolve_mask_path(Path(RAW_DIR), sid, mask_suffix=MASK_SUFFIX)), cv2.IMREAD_GRAYSCALE)
    enc_counts['0/255' if m.max() == 255 else '0/1' if m.max() == 1 else 'blank/other'] += 1
print('\nMask encoding summary (first 50 samples):', dict(enc_counts))

In [ ]:
# ── 3. Size consistency check ────────────────────────────────────────────────
# All images and masks must have the same spatial size for patching to work.

shapes = {}
size_problems = []
for sid in sample_ids[:50]:
    ch_paths = resolve_channel_paths(Path(RAW_DIR), sid, N_SOURCE_CHANNELS)
    mask_path = resolve_mask_path(Path(RAW_DIR), sid, mask_suffix=MASK_SUFFIX)
    im_shape = cv2.imread(str(ch_paths[0]), cv2.IMREAD_GRAYSCALE).shape
    mk_shape  = cv2.imread(str(mask_path),  cv2.IMREAD_GRAYSCALE).shape
    shapes[im_shape] = shapes.get(im_shape, 0) + 1
    if im_shape != mk_shape:
        size_problems.append((sid, im_shape, mk_shape))

print('Image size distribution:', shapes)
if size_problems:
    print(f'\nSize mismatch in {len(size_problems)} samples:')
    for sid, ish, msh in size_problems[:5]:
        print(f'  {sid}: image={ish}, mask={msh}')
else:
    print('OK — image and mask sizes match in all checked samples.')

if len(shapes) > 1:
    print('\nWARNING: mixed image sizes detected.')
    print('PatchConfig.patch_size must be <= the smallest image dimension.')

In [ ]:
# ── 4. Visual spot-check ─────────────────────────────────────────────────────
# Show N_SPOT random samples: first source channel + binarised mask side by side.
# Green overlay = foreground. If nothing is green, mask encoding is wrong.

import random as _random
N_SPOT = 6
spot_ids = _random.sample(sample_ids, min(N_SPOT, len(sample_ids)))

fig, axes = plt.subplots(2, N_SPOT, figsize=(3 * N_SPOT, 6))
for col, sid in enumerate(spot_ids):
    ch_path = resolve_channel_paths(Path(RAW_DIR), sid, N_SOURCE_CHANNELS)[0]
    mk_path = resolve_mask_path(Path(RAW_DIR), sid, mask_suffix=MASK_SUFFIX)
    img = cv2.imread(str(ch_path), cv2.IMREAD_GRAYSCALE)
    msk = binarize(cv2.imread(str(mk_path), cv2.IMREAD_GRAYSCALE))
    fg  = int(msk.sum())

    axes[0, col].imshow(img, cmap='gray');       axes[0, col].set_axis_off()
    axes[0, col].set_title(sid[:18] + '…', fontsize=7)

    # overlay mask in green on top of the image
    overlay = np.stack([img, img, img], axis=-1)
    overlay[msk > 0] = [0, 200, 0]
    axes[1, col].imshow(overlay);                axes[1, col].set_axis_off()
    axes[1, col].set_title(f'fg px: {fg:,}', fontsize=7)

axes[0, 0].set_ylabel('Image', fontsize=9)
axes[1, 0].set_ylabel('Mask (green)', fontsize=9)
plt.suptitle('Real data spot-check — green = foreground label', fontsize=11)
plt.tight_layout()
plt.show()

# PatchConfig guidance based on observed image size
min_dim = min(s[0] for s in shapes) if shapes else 256
print(f'\nSmallest image dimension: {min_dim}px')
print(f'Suggested patch_size range: {min_dim // 4} – {min_dim}')
print(f'Suggested stride: patch_size // 2  (50% overlap, ensures no target sits on a hard edge)')

In [ ]:
shapes

In [ ]:
# ── 5. Suggested PatchConfig ─────────────────────────────────────────────────
# Printed here so you can copy-paste into Step 3 with confidence.

min_dim = min(s[0] for s in shapes) if shapes else 256
patch_size = min(256, min_dim)
stride     = patch_size // 2

# Foreground pixels per patch: rough estimate from a few masks
fg_list = []
for sid in sample_ids[:10]:
    m = binarize(cv2.imread(str(resolve_mask_path(Path(RAW_DIR), sid, mask_suffix=MASK_SUFFIX)), cv2.IMREAD_GRAYSCALE))
    area = m.sum()
    n_patches_est = max(1, (m.shape[0] // stride) * (m.shape[1] // stride))
    fg_list.append(int(area / n_patches_est))
avg_fg = int(np.median(fg_list)) if fg_list else 0

print('── Suggested PatchConfig ──────────────────────────────')
print(f'patch_size      = {patch_size}')
print(f'stride          = {stride}   (50% overlap)')
print(f'min_fg_pixels   = {max(10, avg_fg // 4)}   (¼ of estimated avg fg per patch; lower if targets are tiny)')
print(f'max_empty_frac  = 0.10  (keep 10% of background patches)')
print(f'val_fraction    = 0.20  (20% of SOURCE images for val)')
print()
print('Copy the values above into Step 3 — PatchConfig(...).')

In [ ]:
sid

---
## Step 2 — Inspect raw data and the model-input recipe

Load source channel(s), build the **model input** with `build_image`, and verify the mask.  
For `two_channel`, compare derived-channel candidates before picking a recipe.

In [ ]:
from segmentation_tools_pt.dataset_build import resolve_channel_paths, resolve_mask_path

#sid = 'sample_00'
sid = '2826120010_47801MID_PASS_XRAY_GS_58337'
src_paths = resolve_channel_paths(RAW_DIR, sid, N_SOURCE_CHANNELS)
raw_mask = cv2.imread(str(resolve_mask_path(RAW_DIR, sid, mask_suffix=MASK_SUFFIX)), cv2.IMREAD_GRAYSCALE)

source_chans = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE) for p in src_paths]
for i, ch in enumerate(source_chans):
    print(f'source ch{i}: {ch.shape}, range [{ch.min()}, {ch.max()}]')
print(f'mask unique values: {np.unique(raw_mask).tolist()}')

model_input = build_image(source_chans, CHANNEL_NAMES)
print(f'model input shape (H,W,C): {model_input.shape}  ← C must equal IN_CHANNELS={IN_CHANNELS}')

In [ ]:
# Visualise every channel that goes into the model
fig, axes = plt.subplots(1, len(CHANNEL_NAMES), figsize=(4 * len(CHANNEL_NAMES), 4))
if len(CHANNEL_NAMES) == 1:
    axes = [axes]
for ax, name in zip(axes, CHANNEL_NAMES):
    idx = CHANNEL_NAMES.index(name)
    ax.imshow(model_input[..., idx], cmap='gray')
    ax.set_title(name); ax.set_axis_off()
plt.suptitle(f'Model input channels ({DATA_MODE})')
plt.tight_layout(); plt.show()

# Optional: for two-channel sources, compare derived CH2 candidates before committing
if N_SOURCE_CHANNELS == 2:
    ch0, ch1 = source_chans[0], source_chans[1]
    candidates = {
        'product'    : CHANNEL_OPS['product']([ch0, ch1]),
        'linear_diff': CHANNEL_OPS['linear_diff']([ch0, ch1]),
        'abs_diff'   : CHANNEL_OPS['abs_diff']([ch0, ch1]),
    }
    fig, axes = plt.subplots(1, len(candidates), figsize=(4 * len(candidates), 4))
    for ax, (name, img) in zip(axes, candidates.items()):
        ax.imshow(img, cmap='gray'); ax.set_title(name); ax.set_axis_off()
    plt.suptitle('Derived channel candidates (two-channel source only)')
    plt.tight_layout(); plt.show()

In [ ]:
# Optionally clean up the raw mask with rotated-rectangle fitting.
# For synthetic data this is a no-op; on real hand-drawn masks it removes ragged pixels.
#clean_mask = fit_rotated_rects_mask(binarize(raw_mask))
clean_mask = fit_rotated_rects_mask(raw_mask)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
ax1.imshow(binarize(raw_mask), cmap='gray'); ax1.set_title('Raw mask (binarized)');   ax1.axis('off')
ax2.imshow(clean_mask,         cmap='gray'); ax2.set_title('Cleaned mask (rotated-rect fit)'); ax2.axis('off')
plt.tight_layout()
plt.show()
print(f'Raw foreground px: {int(binarize(raw_mask).sum()):,}  |  Clean: {int(clean_mask.sum()):,}')

---
## Step 3 — Build the patch dataset

`build_patch_dataset` does three things in one call:
1. **Splits** source images into train / val at the source level (never at the patch level — that would leak near-identical crops).
2. **Tiles** each source image into overlapping patches and discards most empty ones.
3. **Writes** `train/`, `val/`, `train.txt`, `val.txt`, `metadata.json` to disk.

In [ ]:
patch_cfg = PatchConfig(
    patch_size=256,
    stride=128,
    min_fg_pixels=206,
    max_empty_fraction=0.10,
    val_fraction=0.20,
    seed=42,
)


In [ ]:
WORK_DIR = Path(r'/path/to/dataset/segmentation')

In [ ]:
PATCH_DIR = WORK_DIR / 'patches'


In [ ]:
Path(PATCH_DIR).resolve()

In [ ]:
Path(PATCH_DIR).mkdir(parents=True, exist_ok=True)

In [ ]:
RAW_DIR

In [ ]:
PATCH_DIR

In [ ]:
MASK_SUFFIX

In [ ]:
N_SOURCE_CHANNELS

In [ ]:
CFG.num_classes

In [ ]:
meta = build_patch_dataset(
    source_dir=str(RAW_DIR),
    output_dir=str(PATCH_DIR),
    cfg=patch_cfg,
    num_classes=CFG.num_classes,
    n_channels=N_SOURCE_CHANNELS,
    n_jobs=1,
    fit_rects=True,
    mask_suffix=MASK_SUFFIX,
)


In [ ]:
len(meta['train'])

In [ ]:

print(f"Train patches : {len(meta['train'])}")
print(f"Val   patches : {len(meta['val'])}")

In [ ]:
meta['train']

In [ ]:
import pandas as pd
df = pd.DataFrame(meta['train'])
print(df[['sample_id', 'source_id', 'fg_pixels', 'is_empty']].head(6).to_string(index=False))
print(f"\nEmpty patches : {df['is_empty'].sum()} / {len(df)}")
print(f"fg_pixels stats:\n{df['fg_pixels'].describe().round(1)}")

# Quick look at one saved patch (patches always store _ch0, _ch1, … on disk)
if len(meta['train']):
    pid = meta['train'][0]['sample_id']
    pimg = cv2.imread(str(PATCH_DIR / 'train' / 'images' / f'{pid}_ch0.png'), cv2.IMREAD_GRAYSCALE)
    pmsk = cv2.imread(str(PATCH_DIR / 'train' / 'masks'  / f'{pid}_mask.png'), cv2.IMREAD_GRAYSCALE)
    print(f'\nPatch {pid}: image {pimg.shape}, mask fg px={(pmsk>0).sum()}')

---
## Step 4 — Compute per-channel statistics

The albumentations `Normalize` layer needs the mean and std of the final model input (after channel synthesis), not of the raw source images. `compute_dataset_stats` streams over the dataset images so it scales to large datasets.

In [ ]:
def _iter_train_images(patch_dir, split='train'):
    """Yield assembled (H, W, C) model inputs from saved patch source channels."""
    img_dir = Path(patch_dir) / split / 'images'
    meta_j = json.loads((Path(patch_dir) / 'metadata.json').read_text())
    for m in meta_j[split]:
        sid = m['sample_id']
        chans = [cv2.imread(str(img_dir / f'{sid}_ch{i}.png'), cv2.IMREAD_GRAYSCALE)
                 for i in range(N_SOURCE_CHANNELS)]
        if any(c is None for c in chans):
            continue
        yield build_image(chans, CHANNEL_NAMES)


In [ ]:
PATCH_DIR.resolve()


In [ ]:

MEAN, STD = compute_dataset_stats(_iter_train_images(PATCH_DIR))
print(f'Channel mean ({len(MEAN)} ch): {[round(v, 4) for v in MEAN]}')
print(f'Channel std  ({len(STD)} ch): {[round(v, 4) for v in STD]}')

---
## Step 5 — Sanity-check the augmented dataset

Before training, always visualise a few augmented samples. `sanity_check` shows image / mask pairs and catches three common bugs:
- Masks not aligned after geometric transforms
- Wrong channel order
- Missing foreground (min_fg_pixels too aggressive)

In [ ]:
train_ds, val_ds = build_datasets(
    root=str(PATCH_DIR),
    num_classes=CFG.num_classes,
    n_source_channels=N_SOURCE_CHANNELS,
    channel_names=CHANNEL_NAMES,
    crop_size=256,
    mean=MEAN,
    std=STD,
)

print(f'Train samples: {len(train_ds)} | Val samples: {len(val_ds)}')
print(f'Image shape  : {tuple(train_ds[0]["image"].shape)}  ← (C,H,W), C={IN_CHANNELS}')
print(f'Mask shape   : {tuple(train_ds[0]["mask"].shape)}')

In [ ]:
# Visually confirm mask-image alignment after augmentation.
# Green = ground truth; the mask should tightly follow the bright rectangular regions.
fig = sanity_check(train_ds, n=6)
plt.show()

---
## Step 6 — Build the model

`in_channels` must match `len(CHANNEL_NAMES)` after `build_image` — **not** `N_SOURCE_CHANNELS` when you derive extra channels (e.g. 2 sources → 3 model inputs).

In [ ]:

model = get_model(
    arch='mobilenet_v2',
    in_channels=CFG.in_channels,
    num_classes=CFG.num_classes,
    pretrained=False,
    #weights_path=f'/path/to/models/mobilenet_v2.pth',

).to(DEVICE)
print(f'{"#"*100}')
print(f' Model is loaded on {DEVICE}')
print(f'{"#"*100}')

In [ ]:


total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model: {total_params:.2f}M params | in_channels={CFG.in_channels}')

dummy = torch.randn(2, CFG.in_channels, 256, 256, device=DEVICE)
with torch.no_grad():
    out = model(dummy)
print(f'Input  : {tuple(dummy.shape)}')
print(f'Output : {tuple(out.shape)}  ← (B, num_classes, H, W)')

---
## Step 7 — Train

We compose the training loop manually (instead of calling the high-level `train()` function) so every piece is visible. This is the recommended approach for research — use the high-level `train()` for production runs where you just want to press go.

**Loss schedule:**

| Phase | Epochs | Tversky | Dice | Boundary | Purpose |
|-------|--------|---------|------|----------|---------|
| 1     | 0–9    | 0.6     | 0.4  | 0.0      | Learn coarse shapes first |
| 2     | 10–34  | 0.5     | 0.3  | 0.2      | Introduce boundary refinement |
| 3     | 35+    | 0.4     | 0.3  | 0.3      | Polish edges |

**Differential learning rates:** the pretrained encoder trains at 1/10 the decoder rate.

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

# Hyperparameters
NUM_EPOCHS   = 20      # keep short for the tutorial; real runs use 60+
BATCH_SIZE   = 8
ENCODER_LR   = 1e-4
DECODER_LR   = 1e-3
CROP_SIZE    = 64
CKPT_DIR     = WORK_DIR / 'checkpoints'
CKPT_DIR.mkdir(exist_ok=True)

# Loaders (WeightedRandomSampler oversamples hard patches)
train_loader, val_loader, train_ds, val_ds = build_dataloaders(
    root=str(PATCH_DIR),
    batch_size=BATCH_SIZE,
    num_workers=0,          # set to 4 on the HPC
    crop_size=CROP_SIZE,
    num_classes=CFG.num_classes,
    n_source_channels=2,
    channel_names=DEFAULT_CHANNELS,
    mean=MEAN, std=STD,
    pin_memory=(DEVICE.type == 'cuda'),
)

criterion = SegLoss(alpha=0.7, beta=0.3, gamma=1.33,
                    w_tversky=0.6, w_dice=0.4, w_boundary=0.0).to(DEVICE)
optimizer = AdamW(get_param_groups(model, ENCODER_LR, DECODER_LR), weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)
scaler    = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None

print(f'Training on {DEVICE} for {NUM_EPOCHS} epochs')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

In [ ]:
# -- Training loop -------------------------------------------------------
# You can interrupt and resume: re-run from the checkpoint cell below.

history = []
best_iou = 0.0

for epoch in range(NUM_EPOCHS):

    # Anneal loss phase
    criterion.set_phase(1 if epoch < 10 else 2 if epoch < 35 else 3)

    # Train
    tr = train_one_epoch(
        model, train_loader, criterion, optimizer, DEVICE,
        scaler=scaler, num_classes=CFG.num_classes,
    )

    # Validate
    va = validate(model, val_loader, criterion, DEVICE, num_classes=CFG.num_classes)

    scheduler.step()

    log = dict(
        epoch=epoch,
        train_loss=tr['loss'],
        val_loss=va['val_loss'],
        val_iou=va['iou'],
        val_dice=va['dice'],
        val_recall=va['recall'],
        lr=scheduler.get_last_lr()[0],
    )
    history.append(log)

    # Save best checkpoint
    if va['iou'] > best_iou:
        best_iou = va['iou']
        save_checkpoint(CKPT_DIR / 'best.pth', epoch, model, optimizer, scheduler, scaler, best_iou, history)

    # Rolling latest checkpoint (enables resume)
    save_checkpoint(CKPT_DIR / 'latest.pth', epoch, model, optimizer, scheduler, scaler, best_iou, history)

    print(
        f'Ep {epoch:02d}/{NUM_EPOCHS-1} | '
        f'loss {log["train_loss"]:.4f} | val_iou {log["val_iou"]:.4f} | '
        f'recall {log["val_recall"]:.4f} | lr {log["lr"]:.2e}'
        + ('  ← best' if va['iou'] == best_iou else '')
    )

print(f'\nFinished. Best val IoU: {best_iou:.4f}')

### Resume from a checkpoint

If training was interrupted, reload and continue:

```python
start_epoch, best_iou, history = load_checkpoint(
    CKPT_DIR / 'latest.pth', model, optimizer, scheduler, scaler, DEVICE
)
# Then re-run the epoch loop starting from start_epoch.
```

---
## Step 8 — Evaluate and visualise

Load the best checkpoint and run a thorough evaluation: pixel metrics, object-level detection rate, and visual overlays.

In [ ]:
# Plot training curves
epochs = [h['epoch'] for h in history]
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(epochs, [h['train_loss'] for h in history], label='train')
axes[0].plot(epochs, [h['val_loss']   for h in history], label='val')
axes[0].set(title='Loss', xlabel='epoch'); axes[0].legend()

axes[1].plot(epochs, [h['val_iou']  for h in history], label='IoU')
axes[1].plot(epochs, [h['val_dice'] for h in history], label='Dice')
axes[1].set(title='Validation overlap', xlabel='epoch'); axes[1].legend()

axes[2].plot(epochs, [h['val_recall'] for h in history])
axes[2].set(title='Validation recall (higher = fewer missed targets)', xlabel='epoch')

plt.tight_layout()
plt.show()

In [ ]:
# Load best weights for evaluation
best_ckpt = torch.load(CKPT_DIR / 'best.pth', map_location=DEVICE, weights_only=False)
model.load_state_dict(best_ckpt['model_state'])
model.eval()
print(f'Loaded best checkpoint from epoch {best_ckpt["epoch"]}  (val IoU = {best_ckpt["best_score"]:.4f})')

In [ ]:
# Full epoch metrics on the validation set
val_metrics = validate(model, val_loader, criterion, DEVICE, num_classes=CFG.num_classes)
print('Validation metrics')
print('-' * 30)
for k, v in val_metrics.items():
    print(f'  {k:<14}: {v:.4f}')

In [ ]:
# Object-level metrics: how many distinct targets did we find?
# (pixel IoU is high but you may still miss a whole small object)
from segmentation_tools_pt.metrics import ObjectMetrics

obj_acc = ObjectMetrics(overlap_thresh=0.5)

with torch.no_grad():
    for batch in val_loader:
        images  = batch['image'].to(DEVICE)
        gt_masks = batch['mask'].numpy()
        logits  = model(images)
        probs   = torch.sigmoid(logits)[:, 0].cpu().numpy()  # (B, H, W)
        preds   = (probs >= 0.5).astype(np.uint8)
        for pred, gt in zip(preds, gt_masks):
            obj_acc.update(pred, gt.astype(np.uint8))

od = obj_acc.compute()
print('Object-level detection')
print('-' * 30)
for k, v in od.items():
    print(f'  {k:<16}: {v}')

In [ ]:
# Visual overlays: GT (green), prediction (red), agreement (magenta)
fig = show_batch(model, val_ds, DEVICE, n=6, threshold=0.5)
plt.show()

In [ ]:
# Worst patches: where does the model struggle?
# These are the patches you should look at hardest — they reveal label errors
# and edge cases not covered by augmentation.
fig = show_top_losses(model, val_ds, criterion, DEVICE, k=4)
plt.show()

---
## Step 9 — Find the operating threshold

The default `threshold=0.5` is rarely optimal in production. In inspection, **missing a target (FN) costs far more than a false alarm (FP)**: the grid software can tolerate FP; a missed pin causes a field failure.

`find_operating_threshold` sweeps the sigmoid threshold and finds the **highest** value that still hits your recall target. Higher threshold = fewer false alarms.

In [ ]:
result = find_operating_threshold(
    model, val_loader, DEVICE,
    target_recall=0.95,   # must miss at most 5% of foreground pixels
    overlap_thresh=0.5,   # an object is 'detected' if 50%+ of it is covered
)

print(f"\nRecommended threshold : {result['recommended_threshold']}")
print(f"Achieved recall       : {result['achieved_recall']:.4f}")
print(f"Achieved precision    : {result['achieved_precision']:.4f}")

# Full sweep table
print('\nFull threshold sweep:')
print(f"{'thresh':>8}  {'recall':>8}  {'prec':>8}  {'IoU':>7}  {'det_rate':>8}  {'missed':>7}  {'FA':>6}")
for r in result['sweep']:
    marker = ' <--' if r['threshold'] == result['recommended_threshold'] else ''
    print(f"{r['threshold']:8.2f}  {r['recall']:8.4f}  {r['precision']:8.4f}  "
          f"{r['iou']:7.4f}  {r['detection_rate']:8.4f}  {r['missed']:7d}  {r['false_alarms']:6d}{marker}")

BEST_THRESH = result['recommended_threshold']

In [ ]:
# Compare 0.5 default vs the recommended threshold on one val patch
sample = val_ds[0]
with torch.no_grad():
    probs = predict(model, sample['image'], DEVICE)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
show_pred(sample['image'], sample['mask'], decode(probs, 0.5),        ax=axes[0]); axes[0].set_title('threshold = 0.5 (default)')
show_pred(sample['image'], sample['mask'], decode(probs, BEST_THRESH), ax=axes[1]); axes[1].set_title(f'threshold = {BEST_THRESH} (recommended)')
axes[2].imshow(probs[0].cpu().numpy(), cmap='hot'); axes[2].set_title('raw probability map'); axes[2].axis('off')
plt.suptitle('Effect of threshold choice')
plt.tight_layout()
plt.show()

---
## Step 10 — Export to ONNX

The ONNX file is what production systems load. `export_onnx` traces the model, validates the graph with the `onnx` checker, and (if `onnxruntime` is installed) verifies that ONNX Runtime's output matches PyTorch's within numerical tolerance.

In [ ]:
ONNX_PATH = str(WORK_DIR / 'model.onnx')

export_onnx(
    model,
    output_path=ONNX_PATH,
    in_channels=CFG.in_channels,  # 3
    input_size=CROP_SIZE,          # 64 — must match training crop
    opset=18,
    verify=True,                   # compare ONNX Runtime vs PyTorch numerically
)

import os
size_mb = os.path.getsize(ONNX_PATH) / 1e6
print(f'\nONNX model: {ONNX_PATH}  ({size_mb:.1f} MB)')

In [ ]:
# Run the exported model with ONNX Runtime and compare to PyTorch
try:
    import onnxruntime as ort
    sess = ort.InferenceSession(ONNX_PATH)
    x = sample['image'].unsqueeze(0).numpy()      # (1, 3, 64, 64)
    ort_logits = sess.run(['logits'], {'image': x})[0]   # (1, 1, 64, 64)
    ort_probs  = 1.0 / (1.0 + np.exp(-ort_logits))
    with torch.no_grad():
        pt_probs = torch.sigmoid(model(sample['image'].unsqueeze(0).to(DEVICE))).cpu().numpy()
    max_diff = float(np.abs(ort_probs - pt_probs).max())
    print(f'Max PyTorch vs ONNX-Runtime difference: {max_diff:.6f}  ({"PASSED" if max_diff < 1e-3 else "WARNING"})')
except ImportError:
    print('onnxruntime not installed — run `pip install onnxruntime` to enable this cell.')

---
## Summary

```
choose DATA_MODE + CHANNEL_NAMES
    → build_patch_dataset(n_channels=N_SOURCE_CHANNELS, mask_suffix=...)
    → build_image(source_chans, CHANNEL_NAMES)  → IN_CHANNELS
    → build_dataloaders(n_source_channels=..., channel_names=...)
    → get_model(in_channels=CFG.in_channels)
    → train → threshold → export_onnx
```

### Switching to your real data

| Your data | Set |
|-----------|-----|
| One X-ray per sample (`id.png`) | `DATA_MODE='single'` or `'single_to_3'` |
| Diffuse + specular pair | `DATA_MODE='two_channel'` |
| Three native channels | `DATA_MODE='three_channel'` |
| Masks named `{id}.png` (no `_mask`) | `mask_suffix=''` in preset |

See `02_next_steps.ipynb` for multi-class, TTA, and custom channel ops.
